# FOL Reasoning Benchmarks Demo

## Overview
This notebook demonstrates the **FOL Reasoning Benchmarks** dataset: a standardized collection of 11,001 logical reasoning examples from three major datasets:
- **FOLIO**: Expert-written natural language premises with explicit first-order logic (FOL) annotations
- **ProofWriter**: Logical theories with structured proof trees and entailment labels
- **RuleTaker**: Depth-graded logical entailment tasks requiring multi-hop reasoning

## Goal
This dataset supports training and evaluating neuro-symbolic pipelines that convert natural language text to FOL representations for reasoning with logic solvers (like Prolog). Each example contains:
- `input`: Natural language text with logical content
- `output`: Label (True/False/entailment/not-entailment)
- `metadata_*` fields: FOL annotations, proof depths, task types

## In This Notebook
1. Load the dataset from a GitHub URL (with local fallback)
2. Explore the data structure and schema
3. Analyze example distributions and metadata
4. Validate data integrity
5. Visualize key insights

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Non-pre-installed packages: always install
_pip('jsonschema==4.26.0')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')

In [ ]:
import json
import sys
from pathlib import Path
from collections import defaultdict, Counter
from typing import Any, Dict, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from jsonschema import validate, ValidationError

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-14efe0-resolution-failure-directed-extraction-d/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    """Load data from GitHub URL with local file fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            data = json.loads(response.read().decode())
            print(f"✓ Loaded from GitHub: {GITHUB_DATA_URL}")
            return data
    except Exception as e:
        print(f"  GitHub fetch failed ({type(e).__name__}), trying local file...")
    
    if Path("mini_demo_data.json").exists():
        with open("mini_demo_data.json") as f:
            data = json.load(f)
            print(f"✓ Loaded from local file: mini_demo_data.json")
            return data
    
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local path")

In [ ]:
data = load_data()

## Data Structure & Schema

Each dataset is a collection of **examples**, where each example has:
- `input`: The raw natural language prompt or logical statement
- `output`: The label or answer
- `metadata_*`: Dataset-specific fields (FOL annotations, proof depths, task type)

This schema is **standardized** across all three datasets (FOLIO, ProofWriter, RuleTaker), making them directly comparable for benchmarking.

In [ ]:
# Inspect the overall structure
print("=" * 70)
print("DATASET STRUCTURE")
print("=" * 70)
print(f"Number of dataset groups: {len(data['datasets'])}")
print()

for dataset_group in data['datasets']:
    name = dataset_group['dataset']
    num_examples = len(dataset_group['examples'])
    print(f"Dataset: {name}")
    print(f"  Examples: {num_examples}")
    if num_examples > 0:
        ex = dataset_group['examples'][0]
        print(f"  Fields: {list(ex.keys())}")
    print()

## Sample Examples

Below are one example from each dataset to illustrate the task types and data content.

In [ ]:
# Display one example from each dataset
for dataset_group in data['datasets']:
    name = dataset_group['dataset']
    examples = dataset_group['examples']
    
    if len(examples) > 0:
        ex = examples[0]
        print("\n" + "=" * 70)
        print(f"Dataset: {name}")
        print("=" * 70)
        print(f"\nTask Type: {ex.get('metadata_task_type', 'N/A')}")
        print(f"\nInput (first 300 chars):\n{ex['input'][:300]}...\n")
        print(f"Output: {ex['output']}")
        print(f"\nMetadata:")
        for key, val in ex.items():
            if key.startswith('metadata_'):
                # Truncate long metadata fields
                val_str = str(val)[:100]
                print(f"  {key}: {val_str}")

## Dataset Statistics

Analyze the distribution of labels, task types, and key metadata fields.

In [ ]:
# Aggregate statistics across datasets
all_examples = []
dataset_counts = {}
task_types = Counter()
output_labels = Counter()

for dataset_group in data['datasets']:
    name = dataset_group['dataset']
    examples = dataset_group['examples']
    dataset_counts[name] = len(examples)
    all_examples.extend(examples)
    
    for ex in examples:
        task_types[ex.get('metadata_task_type', 'unknown')] += 1
        output_labels[ex['output']] += 1

print("=" * 70)
print("STATISTICS")
print("=" * 70)
print(f"\nTotal Examples: {len(all_examples)}")
print(f"\nExamples per Dataset:")
for name, count in sorted(dataset_counts.items()):
    print(f"  {name}: {count}")

print(f"\nTask Types:")
for task_type, count in task_types.most_common():
    pct = 100 * count / len(all_examples) if all_examples else 0
    print(f"  {task_type}: {count} ({pct:.1f}%)")

print(f"\nOutput Labels Distribution:")
for label, count in output_labels.most_common():
    pct = 100 * count / len(all_examples) if all_examples else 0
    print(f"  {label}: {count} ({pct:.1f}%)")

## Schema Validation

Verify that all examples conform to the expected schema: each must have `input`, `output`, and at least one metadata field.

In [ ]:
# Define the expected schema
example_schema = {
    "type": "object",
    "required": ["input", "output"],
    "properties": {
        "input": {"type": "string"},
        "output": {"type": ["string", "number", "boolean"]},
    },
    "additionalProperties": True,  # Allow metadata_* fields
}

# Validate all examples
invalid_count = 0
for ex in all_examples:
    try:
        validate(instance=ex, schema=example_schema)
    except ValidationError as e:
        invalid_count += 1
        print(f"Validation error: {e.message}")

if invalid_count == 0:
    print(f"✓ All {len(all_examples)} examples passed schema validation")
else:
    print(f"✗ {invalid_count} examples failed validation")

# Check for metadata fields
metadata_fields = set()
for ex in all_examples:
    metadata_fields.update([k for k in ex.keys() if k.startswith('metadata_')])

print(f"\nMetadata fields across all examples:")
for field in sorted(metadata_fields):
    count = sum(1 for ex in all_examples if field in ex)
    pct = 100 * count / len(all_examples) if all_examples else 0
    print(f"  {field}: {count}/{len(all_examples)} ({pct:.1f}%)")

## Data Visualization

Visualize the distributions of task types and output labels.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Task types distribution
task_names = list(task_types.keys())
task_counts = list(task_types.values())
axes[0].barh(task_names, task_counts, color='steelblue')
axes[0].set_xlabel('Number of Examples')
axes[0].set_title('Task Type Distribution')
axes[0].grid(axis='x', alpha=0.3)

# Plot 2: Output labels distribution
label_names = list(output_labels.keys())
label_counts = list(output_labels.values())
axes[1].bar(label_names, label_counts, color='darkgreen', alpha=0.7)
axes[1].set_ylabel('Number of Examples')
axes[1].set_title('Output Label Distribution')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Visualization complete")

## Summary

This demo notebook shows how to:
1. **Load** the FOL reasoning benchmark dataset from a GitHub URL (with local fallback)
2. **Explore** the data structure: datasets, examples, metadata fields
3. **Analyze** statistics: task type distribution, label balance, metadata coverage
4. **Validate** schema compliance: ensure all examples have required fields
5. **Visualize** key insights: distribution charts

The dataset is ready for:
- Training neuro-symbolic pipelines to convert NL text to FOL
- Benchmarking logical reasoning systems
- Evaluating text-to-logic translation accuracy
- Analyzing multi-hop reasoning difficulty (hop_depth metadata)

For the **full dataset** with all 11,001 examples, replace the GitHub URL with the full data URL and increase processing capacity as needed.

In [ ]:
print("\n" + "=" * 70)
print("NOTEBOOK COMPLETE")
print("=" * 70)
print(f"\n✓ Loaded {len(all_examples)} examples from {len(data['datasets'])} datasets")
print(f"✓ Validated all examples against schema")
print(f"✓ Generated visualizations and statistics")
print(f"\nDataset is ready for:")
print(f"  - Text-to-FOL translation training")
print(f"  - Logical reasoning benchmarking")
print(f"  - Neuro-symbolic pipeline evaluation")